# 00 — Environment check

Run this top to bottom. If every cell executes without error, the lab is ready and you can trust
every later exercise. It proves four things: the python stack imports, DuckDB reads the parquet
snapshots, charts render, and the smoke test (`scripts/env_check.py`) passes.

If a dataset is missing: `make data` from the `lab/` directory, then re-run.

In [ ]:
import sys, duckdb, pandas as pd, matplotlib, seaborn, statsmodels
print("python     ", sys.version.split()[0])
print("duckdb     ", duckdb.__version__)
print("pandas     ", pd.__version__)
print("matplotlib ", matplotlib.__version__)
print("seaborn    ", seaborn.__version__)
print("statsmodels", statsmodels.__version__)

In [ ]:
# DuckDB reads the parquet snapshots directly — no import step, no engine to babysit.
con = duckdb.connect()
con.sql("""
    SELECT assignee,
           count(*)                                   AS tasks,
           count(*) FILTER (WHERE completed_at_utc IS NOT NULL) AS completed,
           round(median(lead_time_hours), 2)          AS median_lead_hours
    FROM '../data/raw/kanban_tasks.parquet'
    GROUP BY 1 ORDER BY tasks DESC
""").df()

In [ ]:
# Chart path: pandas + matplotlib. Weekly completed-task throughput — week 1 material
# (is this leading or lagging? actionable or vanity?).
df = con.sql("""
    SELECT date_trunc('week', completed_at_utc) AS week,
           count(*)                             AS completed_tasks
    FROM '../data/raw/kanban_tasks.parquet'
    WHERE completed_at_utc IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").df()
ax = df.plot(x='week', y='completed_tasks', figsize=(9, 3), marker='o', legend=False)
ax.set_title("Completed tasks per week (snapshot)")
ax.set_ylabel("tasks")
df.tail(6)

In [ ]:
# The authoritative smoke test — same script `make check` runs from the shell.
%run ../scripts/env_check.py